# ATC Round 2 Colab Runner (Isolated Clone)

This notebook clones the submitted repo in an isolated Colab workspace, then runs SFT + GRPO with one top switch.

- `QUICK_T4 = True` (default): low-iteration demo run in minutes.
- `QUICK_T4 = False`: reproducibility mode (episodes=100, no strict roster integrity).

In [ ]:
# 1) Clone submitted repo snapshot (isolation-first)
import os, subprocess, pathlib

REPO_URL = os.environ.get("ATC_REPO_URL", "https://huggingface.co/spaces/YOUR_USER/YOUR_SPACE")
REPO_REF = os.environ.get("ATC_REPO_REF", "")  # commit hash or tag optional
WORKDIR = pathlib.Path('/content/ats-submission')

if WORKDIR.exists():
    subprocess.run(['rm', '-rf', str(WORKDIR)], check=False)

subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
if REPO_REF.strip():
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', REPO_REF], check=True)

print('Repo ready at', WORKDIR)
subprocess.run(['git', '-C', str(WORKDIR), 'rev-parse', 'HEAD'], check=False)


In [ ]:
# 2) Install deps
import subprocess
subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['bash', '-lc', 'cd /content/ats-submission && uv sync --frozen --extra training'], check=True)
print('Dependencies installed')


In [ ]:
# 3) Single top switch
QUICK_T4 = True

if QUICK_T4:
    CFG = {
        'episodes': 20,
        'sft_max_steps': 80,
        'grpo_max_steps': 30,
        'batch_size': 6,
        'grad_accum': 1,
        'n_generations': 2,
        'relax_roster': True,
        'run_eval': False,
    }
else:
    # Repro mode mirrors project flow but with episodes=100 and relaxed roster for portability.
    CFG = {
        'episodes': 100,
        'sft_max_steps': 400,
        'grpo_max_steps': None,
        'batch_size': 12,
        'grad_accum': 2,
        'n_generations': 4,
        'relax_roster': True,
        'run_eval': True,
    }

CFG


In [ ]:
# 4) Run SFT then GRPO
import os, subprocess, textwrap

repo = '/content/ats-submission'
out = '/content/atc_outputs'
os.makedirs(out, exist_ok=True)

env = os.environ.copy()
env['TORCH_COMPILE_DISABLE'] = '1'
env['UNSLOTH_DISABLE_STATISTICS'] = '1'
env['UNSLOTH_DISABLE_FAST_GENERATION'] = '1'
env['ATC_LIVE_PASSES'] = '2.5'
env['ATC_SAVE_STEPS'] = '20'
if CFG['relax_roster']:
    env['ATC_RELAX_ROSTER'] = '1'

sft_cmd = [
    'python', 'training/train_sft.py',
    '--model', 'Qwen/Qwen2.5-7B-Instruct',
    '--output_dir', f'{out}/atc-sft-json',
    '--n_episodes', '120',
    '--max_steps', str(CFG['sft_max_steps']),
    '--batch_size', '4' if not QUICK_T4 else '2',
    '--grad_accum', '2' if not QUICK_T4 else '1',
]
subprocess.run(sft_cmd, cwd=repo, env=env, check=True)

grpo_cmd = [
    'python', 'training/train_grpo.py',
    '--model', 'Qwen/Qwen2.5-7B-Instruct',
    '--output_dir', f'{out}/atc-multiagent',
    '--adapter_in', f'{out}/atc-sft-json',
    '--episodes', str(CFG['episodes']),
    '--grounded_curriculum',
    '--batch_size', str(CFG['batch_size']),
    '--grad_accum', str(CFG['grad_accum']),
    '--n_generations', str(CFG['n_generations']),
    '--eval_episodes', '3',
]
if CFG['grpo_max_steps'] is not None:
    grpo_cmd += ['--max_steps', str(CFG['grpo_max_steps'])]
if not CFG['run_eval']:
    grpo_cmd += ['--no_eval']

subprocess.run(grpo_cmd, cwd=repo, env=env, check=True)

print('Done. Outputs at', out)
